In [26]:
!pip -q install datasets sentence-transformers faiss-cpu rank_bm25 pandas numpy tqdm requests groq

In [27]:
import os, re, json, math, time, textwrap, requests
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from rank_bm25 import BM25Okapi

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

N_EVAL = 100
TOP_K = 5
CANDIDATE_K = 20

EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
RERANKER_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'


In [28]:
popqa = load_dataset('akariasai/PopQA', split='test')
print(popqa)
print(popqa.column_names)
print(popqa[0])

# Convert to pandas for easier handling
df = popqa.to_pandas()
print('Rows:', len(df))
df.head()

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['id', 'subj', 'prop', 'obj', 'subj_id', 'prop_id', 'obj_id', 's_aliases', 'o_aliases', 's_uri', 'o_uri', 's_wiki_title', 'o_wiki_title', 's_pop', 'o_pop', 'question', 'possible_answers'],
    num_rows: 14267
})
['id', 'subj', 'prop', 'obj', 'subj_id', 'prop_id', 'obj_id', 's_aliases', 'o_aliases', 's_uri', 'o_uri', 's_wiki_title', 'o_wiki_title', 's_pop', 'o_pop', 'question', 'possible_answers']
{'id': 4222362, 'subj': 'George Rankin', 'prop': 'occupation', 'obj': 'politician', 'subj_id': 1850297, 'prop_id': 22, 'obj_id': 2834605, 's_aliases': '["George James Rankin"]', 'o_aliases': '["political leader","political figure","polit.","pol"]', 's_uri': 'http://www.wikidata.org/entity/Q5543720', 'o_uri': 'http://www.wikidata.org/entity/Q82955', 's_wiki_title': 'George Rankin', 'o_wiki_title': 'Politician', 's_pop': 142, 'o_pop': 25692, 'question': "What is George Rankin's occupation?", 'possible_answers': '["politician", "political leader", "political figure", "poli

,id,subj,prop,obj,subj_id,prop_id,obj_id,s_aliases,o_aliases,s_uri,o_uri,s_wiki_title,o_wiki_title,s_pop,o_pop,question,possible_answers
0,4222362,George Rankin,occupation,politician,1850297,22,2834605,"[""George James Rankin""]","[""political leader"",""political figure"",""polit....",http://www.wikidata.org/entity/Q5543720,http://www.wikidata.org/entity/Q82955,George Rankin,Politician,142,25692,What is George Rankin's occupation?,"[""politician"", ""political leader"", ""political ..."
1,4725190,John Mayne,occupation,journalist,2079053,22,663400,[],"[""journo"",""journalists""]",http://www.wikidata.org/entity/Q6247345,http://www.wikidata.org/entity/Q1930187,John Mayne,Journalist,236,24952,What is John Mayne's occupation?,"[""journalist"", ""journo"", ""journalists""]"
2,4382392,Henry Feilden,occupation,politician,1925450,22,2834605,"[""Henry Master Feilden""]","[""political leader"",""political figure"",""polit....",http://www.wikidata.org/entity/Q5725578,http://www.wikidata.org/entity/Q82955,Henry Feilden (Conservative politician),Politician,58,25692,What is Henry Feilden's occupation?,"[""politician"", ""political leader"", ""political ..."
3,4822110,Kathy Saltzman,occupation,politician,2122743,22,2834605,[],"[""political leader"",""political figure"",""polit....",http://www.wikidata.org/entity/Q6377295,http://www.wikidata.org/entity/Q82955,Kathy Saltzman,Politician,127,25692,What is Kathy Saltzman's occupation?,"[""politician"", ""political leader"", ""political ..."
4,4011112,Eleanor Davis,occupation,cartoonist,1752619,22,68412,"[""Eleanor McCutcheon Davis""]","[""graphic artist"",""animator"",""illustrator""]",http://www.wikidata.org/entity/Q5354261,http://www.wikidata.org/entity/Q1114448,Eleanor Davis,Cartoonist,317,9649,What is Eleanor Davis's occupation?,"[""cartoonist"", ""graphic artist"", ""animator"", ""..."


In [29]:
# Use a fixed random sample so the experiment is reproducible.
eval_df = df.sample(n=N_EVAL, random_state=RANDOM_SEED).reset_index(drop=True)

# Normalize possible answers. Some versions store it as a list; some as string/list-like.
def normalize_answers(row):
    answers = []
    if 'possible_answers' in row and row['possible_answers'] is not None:
        pa = row['possible_answers']
        if isinstance(pa, list):
            answers.extend(pa)
        elif isinstance(pa, str):
            try:
                parsed = json.loads(pa)
                if isinstance(parsed, list): answers.extend(parsed)
                else: answers.append(str(parsed))
            except Exception:
                answers.append(pa)
    if 'obj' in row and row['obj'] is not None:
        answers.append(str(row['obj']))
    # unique clean answers
    clean = []
    for a in answers:
        a = str(a).strip()
        if a and a.lower() not in [x.lower() for x in clean]:
            clean.append(a)
    return clean

eval_df['answers'] = eval_df.apply(normalize_answers, axis=1)
print(eval_df[['id','question','subj','prop','obj','answers']].head())


        id                                           question  \
0  2435555          Who was the screenwriter for On the Town?   
1  5435111  What is the religion of Juan Soldevilla y Romero?   
2  2490460                            What genre is Haddaway?   
3   402939               Who was the composer of Chances Are?   
4  5947997              Who was the producer of The Pioneers?   

                       subj          prop               obj  \
0               On the Town  screenwriter      Betty Comden   
1  Juan Soldevilla y Romero      religion   Catholic Church   
2                  Haddaway         genre         Eurodance   
3               Chances Are      composer     Maurice Jarre   
4              The Pioneers      producer  Franklyn Barrett   

                                             answers  
0          [Betty Comden, Basya Cohen, Adolph Green]  
1  [Catholic Church, Roman Catholic Church, Churc...  
2                [Eurodance, Euro-dance, Euro dance]  
3         

In [30]:
def fetch_wikipedia_summary(title):
    if title is None or str(title).strip() == '':
        return None
    safe_title = str(title).replace(' ', '_')
    url = f'https://en.wikipedia.org/api/rest_v1/page/summary/{safe_title}'
    try:
        r = requests.get(url, timeout=10, headers={'User-Agent':'CSAI422-RAG-Assignment/1.0'})
        if r.status_code == 200:
            data = r.json()
            text = data.get('extract', '')
            page_url = data.get('content_urls', {}).get('desktop', {}).get('page', url)
            if len(text.split()) > 20:
                return {'title': title, 'text': text, 'source': page_url}
    except Exception:
        pass
    return None

def chunk_text(text, max_words=100, overlap=25):
    words = text.split()
    chunks=[]
    start=0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunk = ' '.join(words[start:end])
        if len(chunk.split()) >= 20:
            chunks.append(chunk)
        if end == len(words): break
        start += max_words - overlap
    return chunks

# Build titles: target subjects + distractors
subject_titles = eval_df['s_wiki_title'].dropna().astype(str).tolist()
distractor_titles = df.sample(n=300, random_state=RANDOM_SEED+1)['s_wiki_title'].dropna().astype(str).tolist()
all_titles = list(dict.fromkeys(subject_titles + distractor_titles))

raw_docs=[]
for title in tqdm(all_titles, desc='Fetching Wikipedia summaries'):
    doc = fetch_wikipedia_summary(title)
    if doc:
        raw_docs.append(doc)
    time.sleep(0.05)

passages=[]
for i, doc in enumerate(raw_docs):
    for j, chunk in enumerate(chunk_text(doc['text'])):
        passages.append({
            'pid': f'P{i:04d}_{j:02d}',
            'title': doc['title'],
            'source': doc['source'],
            'text': chunk
        })

corpus_df = pd.DataFrame(passages)
print('Raw docs:', len(raw_docs))
print('Passages:', len(corpus_df))
corpus_df.head()

Fetching Wikipedia summaries:   0%|          | 0/395 [00:00<?, ?it/s]

Raw docs: 49
Passages: 57


,pid,title,source,text
0,P0000_00,On the Town (film),https://en.wikipedia.org/wiki/On_the_Town_(film),On the Town is a 1949 American Technicolor mus...
1,P0001_00,Juan Soldevilla y Romero,https://en.wikipedia.org/wiki/Juan_Soldevila_y...,Juan Soldevila y Romero was a Spanish Cardinal...
2,P0002_00,The Album (Haddaway album),https://en.wikipedia.org/wiki/The_Album_(Hadda...,The Album is the debut album by Trinidadian-Ge...
3,P0003_00,Chances Are (film),https://en.wikipedia.org/wiki/Chances_Are_(film),Chances Are is a 1989 American romantic fantas...
4,P0004_00,The City (1916 film),https://en.wikipedia.org/wiki/The_City_(1916_f...,The City is a lost 1916 silent film based on C...


### Evidence that passage identifiers and metadata are preserved

In [31]:
corpus_df[['pid','title','source','text']].head(10)

,pid,title,source,text
0,P0000_00,On the Town (film),https://en.wikipedia.org/wiki/On_the_Town_(film),On the Town is a 1949 American Technicolor mus...
1,P0001_00,Juan Soldevilla y Romero,https://en.wikipedia.org/wiki/Juan_Soldevila_y...,Juan Soldevila y Romero was a Spanish Cardinal...
2,P0002_00,The Album (Haddaway album),https://en.wikipedia.org/wiki/The_Album_(Hadda...,The Album is the debut album by Trinidadian-Ge...
3,P0003_00,Chances Are (film),https://en.wikipedia.org/wiki/Chances_Are_(film),Chances Are is a 1989 American romantic fantas...
4,P0004_00,The City (1916 film),https://en.wikipedia.org/wiki/The_City_(1916_f...,The City is a lost 1916 silent film based on C...
5,P0005_00,Sixteen Candles,https://en.wikipedia.org/wiki/Sixteen_Candles,Sixteen Candles is a 1984 American coming-of-a...
6,P0006_00,The Great Fire (TV series),https://en.wikipedia.org/wiki/The_Great_Fire_(...,The Great Fire is a four-part television drama...
7,P0007_00,As You Like It (1936 film),https://en.wikipedia.org/wiki/As_You_Like_It_(...,As You Like It is a 1936 British romantic come...
8,P0008_00,Rise (Taeyang album),https://en.wikipedia.org/wiki/Rise_(Taeyang_al...,Rise is the second studio album by South Korea...
9,P0008_01,Rise (Taeyang album),https://en.wikipedia.org/wiki/Rise_(Taeyang_al...,album also debuted at number one on the Gaon A...


In [32]:
embedder = SentenceTransformer(EMBED_MODEL_NAME)

passage_texts = corpus_df['text'].tolist()
passage_embeddings = embedder.encode(passage_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
passage_embeddings = np.asarray(passage_embeddings, dtype='float32')

# Cosine similarity via normalized vectors + inner product
index = faiss.IndexFlatIP(passage_embeddings.shape[1])
index.add(passage_embeddings)
print('FAISS index size:', index.ntotal)
print('Embedding dimension:', passage_embeddings.shape[1])

def dense_retrieve(query, k=TOP_K):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, idxs = index.search(q_emb, k)
    results=[]
    for score, idx in zip(scores[0], idxs[0]):
        row = corpus_df.iloc[int(idx)].to_dict()
        row['score'] = float(score)
        results.append(row)
    return results

# Show baseline retrieval examples for 3 queries
for q in eval_df['question'].head(3):
    print('\nQUESTION:', q)
    for r in dense_retrieve(q, k=3):
        print(r['pid'], round(r['score'],3), r['title'], '-', r['text'][:180])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index size: 57
Embedding dimension: 384

QUESTION: Who was the screenwriter for On the Town?
P0000_00 0.512 On the Town (film) - On the Town is a 1949 American Technicolor musical film with music by Leonard Bernstein and Roger Edens and book and lyrics by Betty Comden and Adolph Green. It is an adaptation of
P0004_00 0.441 The City (1916 film) - The City is a lost 1916 silent film based on Clyde Fitch's 1909 play, The City. It was distributed by the World Film Company.
P0029_00 0.41 After Hours (film) - After Hours is a 1985 American neo-noir black comedy film directed by Martin Scorsese, written by Joseph Minion, and produced by Amy Robinson, Griffin Dunne, and Robert F. Colesber

QUESTION: What is the religion of Juan Soldevilla y Romero?
P0001_00 0.669 Juan Soldevilla y Romero - Juan Soldevila y Romero was a Spanish Cardinal of the Roman Catholic Church who served as Archbishop of Zaragoza from 1901 until his death, and was elevated to the rank of cardinal
P0013_00 0.294 Peter

In [33]:
def norm_text(s):
    return re.sub(r'[^a-z0-9 ]+', ' ', str(s).lower()).strip()

def is_relevant(passage_text, answers):
    p = norm_text(passage_text)
    for a in answers:
        a_norm = norm_text(a)
        if a_norm and a_norm in p:
            return True
    return False

def evaluate_retriever(retrieve_fn, data=eval_df, k_values=(1,3,5), name='system'):
    rows=[]
    mrrs=[]
    for _, ex in tqdm(data.iterrows(), total=len(data), desc=f'Evaluating {name}'):
        results = retrieve_fn(ex['question'], k=max(k_values))
        rels = [is_relevant(r['text'], ex['answers']) for r in results]
        row={'id':ex['id'], 'question':ex['question']}
        for k in k_values:
            top = rels[:k]
            row[f'Recall@{k}'] = 1.0 if any(top) else 0.0
            row[f'Precision@{k}'] = sum(top)/k
        rr = 0.0
        for rank, rel in enumerate(rels, start=1):
            if rel:
                rr = 1.0/rank
                break
        row['MRR'] = rr
        rows.append(row)
        mrrs.append(rr)
    detail = pd.DataFrame(rows)
    metrics = {col: detail[col].mean() for col in detail.columns if col.startswith('Recall') or col.startswith('Precision')}
    metrics['MRR'] = np.mean(mrrs)
    return pd.DataFrame([metrics], index=[name]), detail

baseline_metrics, baseline_detail = evaluate_retriever(dense_retrieve, name='Dense baseline')
baseline_metrics

Evaluating Dense baseline:   0%|          | 0/100 [00:00<?, ?it/s]

,Recall@1,Precision@1,Recall@3,Precision@3,Recall@5,Precision@5,MRR
Dense baseline,0.27,0.27,0.3,0.116667,0.34,0.102,0.295


In [34]:
def parse_aliases(x):
    if x is None: return []
    if isinstance(x, list): return [str(i) for i in x]
    if isinstance(x, str):
        try:
            parsed = json.loads(x)
            if isinstance(parsed, list): return [str(i) for i in parsed]
        except Exception:
            pass
        return [x]
    return []

prop_keywords = {
    'occupation': 'job profession career occupation',
    'place of birth': 'born birthplace birth place',
    'country': 'country nation nationality',
    'author': 'written by author writer',
    'director': 'directed by director film',
    'composer': 'music composer composed by',
    'genre': 'genre style category',
    'located in': 'located in place location'
}

def expand_query_from_row(row):
    q = row['question']
    additions=[]
    additions.append(str(row.get('subj','')))
    aliases = parse_aliases(row.get('s_aliases', []))[:3]
    additions.extend(aliases)
    prop = str(row.get('prop','')).lower()
    for key, val in prop_keywords.items():
        if key in prop:
            additions.append(val)
    additions = [a for a in additions if a and str(a).lower() not in q.lower()]
    return q + ' ' + ' '.join(additions)

expanded_examples=[]
for _, row in eval_df.head(8).iterrows():
    expanded_examples.append({'original': row['question'], 'expanded': expand_query_from_row(row)})
pd.DataFrame(expanded_examples)

,original,expanded
0,Who was the screenwriter for On the Town?,Who was the screenwriter for On the Town?
1,What is the religion of Juan Soldevilla y Romero?,What is the religion of Juan Soldevilla y Rome...
2,What genre is Haddaway?,What genre is Haddaway? The Album genre style ...
3,Who was the composer of Chances Are?,Who was the composer of Chances Are? music com...
4,Who was the producer of The Pioneers?,Who was the producer of The Pioneers?
5,Who was the screenwriter for The City?,Who was the screenwriter for The City?
6,In what city was Jerrold Katz born?,In what city was Jerrold Katz born? Jerrold Ja...
7,Who was the screenwriter for Sixteen Candles?,Who was the screenwriter for Sixteen Candles? ...


In [35]:
# Need a lookup from question to row for expansion inside retriever
question_to_row = {row['question']: row for _, row in eval_df.iterrows()}

def dense_expanded_retrieve(query, k=TOP_K):
    row = question_to_row.get(query)
    expanded = expand_query_from_row(row) if row is not None else query
    return dense_retrieve(expanded, k=k)

expanded_metrics, expanded_detail = evaluate_retriever(dense_expanded_retrieve, name='Dense + query expansion')
pd.concat([baseline_metrics, expanded_metrics])

Evaluating Dense + query expansion:   0%|          | 0/100 [00:00<?, ?it/s]

,Recall@1,Precision@1,Recall@3,Precision@3,Recall@5,Precision@5,MRR
Dense baseline,0.27,0.27,0.30,0.116667,0.34,0.102,0.295000
Dense + query expansion,0.28,0.28,0.32,0.133333,0.34,0.100,0.303333


In [36]:
tokenized_corpus = [norm_text(t).split() for t in corpus_df['text'].tolist()]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query, k=CANDIDATE_K):
    tokens = norm_text(query).split()
    scores = bm25.get_scores(tokens)
    idxs = np.argsort(scores)[::-1][:k]
    results=[]
    for idx in idxs:
        row = corpus_df.iloc[int(idx)].to_dict()
        row['score'] = float(scores[idx])
        results.append(row)
    return results

def rrf_fuse(result_lists, k=TOP_K, rrf_k=60):
    scores = defaultdict(float)
    pid_to_row = {}
    for results in result_lists:
        for rank, r in enumerate(results, start=1):
            scores[r['pid']] += 1.0 / (rrf_k + rank)
            pid_to_row[r['pid']] = r
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    output=[]
    for pid, score in ranked:
        row = dict(pid_to_row[pid])
        row['score'] = float(score)
        output.append(row)
    return output

def hybrid_retrieve(query, k=TOP_K):
    row = question_to_row.get(query)
    expanded = expand_query_from_row(row) if row is not None else query
    dense_results = dense_retrieve(expanded, k=CANDIDATE_K)
    bm25_results = bm25_retrieve(expanded, k=CANDIDATE_K)
    return rrf_fuse([dense_results, bm25_results], k=k)

for q in eval_df['question'].head(2):
    print('\nQUESTION:', q)
    for r in hybrid_retrieve(q, k=5):
        print(r['pid'], round(r['score'],4), r['title'], '-', r['text'][:150])

hybrid_metrics, hybrid_detail = evaluate_retriever(hybrid_retrieve, name='Hybrid RRF')
pd.concat([baseline_metrics, expanded_metrics, hybrid_metrics])


QUESTION: Who was the screenwriter for On the Town?
P0000_00 0.0328 On the Town (film) - On the Town is a 1949 American Technicolor musical film with music by Leonard Bernstein and Roger Edens and book and lyrics by Betty Comden and Adolph
P0041_00 0.0318 The Company (miniseries) - The Company is a three-part serial about the activities of the CIA during the Cold War. It was based on the best-selling 2002 novel of the same name b
P0004_00 0.0289 The City (1916 film) - The City is a lost 1916 silent film based on Clyde Fitch's 1909 play, The City. It was distributed by the World Film Company.
P0022_00 0.0276 Grease (film) - Grease is a 1978 American musical romantic comedy film directed by Randal Kleiser from a screenplay by Bronté Woodard and an adaptation by co-producer
P0045_01 0.0268 Argo (2012 film) - Mendez led the rescue of six U.S. diplomats from Tehran, Iran, under the guise of filming a science-fiction film during the 1979–81 Iran hostage crisi

QUESTION: What is the religion

Evaluating Hybrid RRF:   0%|          | 0/100 [00:00<?, ?it/s]

,Recall@1,Precision@1,Recall@3,Precision@3,Recall@5,Precision@5,MRR
Dense baseline,0.27,0.27,0.30,0.116667,0.34,0.102,0.295000
Dense + query expansion,0.28,0.28,0.32,0.133333,0.34,0.100,0.303333
Hybrid RRF,0.26,0.26,0.33,0.130000,0.35,0.102,0.296167


## 9. Reranking

The cross-encoder scores each `(question, passage)` pair and reranks the top hybrid candidates.


In [37]:
reranker = CrossEncoder(RERANKER_NAME)

def hybrid_candidates(query, candidate_k=CANDIDATE_K):
    row = question_to_row.get(query)
    expanded = expand_query_from_row(row) if row is not None else query
    dense_results = dense_retrieve(expanded, k=candidate_k)
    bm25_results = bm25_retrieve(expanded, k=candidate_k)
    return rrf_fuse([dense_results, bm25_results], k=candidate_k)

def reranked_retrieve(query, k=TOP_K):
    candidates = hybrid_candidates(query, candidate_k=CANDIDATE_K)
    pairs = [(query, c['text']) for c in candidates]
    scores = reranker.predict(pairs)
    for c, s in zip(candidates, scores):
        c['rerank_score'] = float(s)
        c['score'] = float(s)
    return sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)[:k]

# Before/after example
q = eval_df.loc[0, 'question']
print('QUESTION:', q)
print('\nBefore rerank:')
for r in hybrid_candidates(q, candidate_k=5):
    print(r['pid'], r['title'], r['text'][:120])
print('\nAfter rerank:')
for r in reranked_retrieve(q, k=5):
    print(r['pid'], round(r['rerank_score'],3), r['title'], r['text'][:120])

reranked_metrics, reranked_detail = evaluate_retriever(reranked_retrieve, name='Hybrid + reranker')
comparison_retrieval = pd.concat([baseline_metrics, expanded_metrics, hybrid_metrics, reranked_metrics])
comparison_retrieval

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUESTION: Who was the screenwriter for On the Town?

Before rerank:
P0000_00 On the Town (film) On the Town is a 1949 American Technicolor musical film with music by Leonard Bernstein and Roger Edens and book and lyr
P0041_00 The Company (miniseries) The Company is a three-part serial about the activities of the CIA during the Cold War. It was based on the best-selling
P0004_00 The City (1916 film) The City is a lost 1916 silent film based on Clyde Fitch's 1909 play, The City. It was distributed by the World Film Com
P0029_00 After Hours (film) After Hours is a 1985 American neo-noir black comedy film directed by Martin Scorsese, written by Joseph Minion, and pro
P0040_00 Seven Songs for Malcolm X Seven Songs for Malcolm X is a British documentary film about the life of Malcolm X, the influential civil rights activi

After rerank:
P0000_00 6.085 On the Town (film) On the Town is a 1949 American Technicolor musical film with music by Leonard Bernstein and Roger Edens and book and lyr
P0

Evaluating Hybrid + reranker:   0%|          | 0/100 [00:00<?, ?it/s]

,Recall@1,Precision@1,Recall@3,Precision@3,Recall@5,Precision@5,MRR
Dense baseline,0.27,0.27,0.30,0.116667,0.34,0.102,0.295000
Dense + query expansion,0.28,0.28,0.32,0.133333,0.34,0.100,0.303333
Hybrid RRF,0.26,0.26,0.33,0.130000,0.35,0.102,0.296167
Hybrid + reranker,0.32,0.32,0.35,0.146667,0.36,0.112,0.334167


## 10. Citation-grounded answer generation

This section uses Groq if you provide `GROQ_API_KEY`. If no key is available, it falls back to a simple extractive answer using the known PopQA answer for demonstration. For final submission, use Groq or another LLM and include its generated answers.


In [38]:
GROUNDING_PROMPT = '''You are a factual question answering assistant.
Answer the question using ONLY the provided passages.
Every factual claim must be supported by inline citations such as [P1] or [P2].
Do NOT invent information not found in the passages.
If the passages do not provide enough evidence, reply exactly:
"Insufficient evidence in the retrieved passages."
Keep the answer brief (1-3 sentences).

Question: {question}

Passages:
{passages}

Answer:'''

# Task 3.2: Reflection prompt
# The reflection stage critiques the draft answer on three axes:
#   (a) grounding  - every claim is traceable to a cited passage
#   (b) citation format - [P1]-style tags are present and correct
#   (c) completeness - the answer actually addresses the question
# If the draft fails any check, the model rewrites using the same evidence.
# Abstention is always preferred over fabrication.
REFLECTION_PROMPT = '''You are a rigorous fact-checker for a RAG system.

Question: {question}

Retrieved passages (the ONLY allowed evidence):
{passages}

Draft answer: {answer}

Perform the following checks then produce a final answer:
1. GROUNDING: Is every factual claim directly supported by the passages above?
   If a claim has no supporting passage, remove it.
2. CITATIONS: Does the draft contain at least one [P#] citation?
   If not, add the correct citation tag(s) from the passage list.
3. COMPLETENESS: Does the answer actually address the question?
   If not, fix it using only the passages above.
4. ABSTAIN: If no passage supports a confident answer, write exactly:
   "Insufficient evidence in the retrieved passages."

Return ONLY the final corrected answer, nothing else.
Final answer:'''

def format_passages(results):
    lines = []
    for i, r in enumerate(results, start=1):
        r['citation'] = f'P{i}'
        snippet = r['text'][:500].replace('\n', ' ')
        lines.append(f'[{r["citation"]}] pid={r["pid"]}, title={r["title"]}: {snippet}')
    return '\n'.join(lines)

def call_groq(prompt, model='llama-3.1-8b-instant'):
    # FIX: use the key directly as a value, not as an env-var name
    api_key = os.environ.get('GROQ_API_KEY')
    if not api_key:
        return None
    from groq import Groq
    client = Groq(api_key=api_key)
    resp = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=300,
    )
    return resp.choices[0].message.content.strip()

def generate_grounded_answer(question, results, known_answers=None):
    passages = format_passages(results)
    prompt = GROUNDING_PROMPT.format(question=question, passages=passages)
    answer = call_groq(prompt)
    if answer is None:
        supported = []
        for i, r in enumerate(results, start=1):
            if known_answers and is_relevant(r['text'], known_answers):
                supported.append(f'[P{i}]')
        if supported and known_answers:
            answer = f"The answer is {known_answers[0]} {' '.join(supported[:2])}."
        else:
            answer = 'Insufficient evidence in the retrieved passages.'
    return answer, passages

def reflect_answer(question, passages, draft_answer):
    prompt = REFLECTION_PROMPT.format(question=question, passages=passages, answer=draft_answer)
    revised = call_groq(prompt)
    if revised is None:
        has_citation = bool(re.search(r'\[P\d+\]', draft_answer))
        insufficient = 'Insufficient evidence' in draft_answer
        if not has_citation and not insufficient:
            return 'Insufficient evidence in the retrieved passages. [reflection: no citations in draft]'
        return draft_answer
    return revised

print('Generation + reflection functions defined.')
print("NOTE: set os.environ['GROQ_API_KEY'] = 'gsk_...' before running generation cells.")


## Task 3.1 - Generate 10 citation-grounded answers

Each row shows: question | gold answers | retrieved passage IDs | draft answer (pre-reflection) | final answer (post-reflection).

**Prompt design (Task 3.2):** `GROUNDING_PROMPT` instructs the model to:
- cite every claim with `[P#]` tags tied to the passage list,
- abstain with a fixed phrase rather than fabricate,
- stay brief (1-3 sentences) to avoid hallucination drift.

`REFLECTION_PROMPT` then verifies grounding, citation presence, and completeness.

In [39]:
qa_examples=[]
for _, ex in tqdm(eval_df.head(10).iterrows(), total=10):
    results = reranked_retrieve(ex['question'], k=5)
    draft, passage_block = generate_grounded_answer(ex['question'], results, known_answers=ex['answers'])
    final = reflect_answer(ex['question'], passage_block, draft)
    qa_examples.append({
        'id': ex['id'],
        'question': ex['question'],
        'gold_answers': ex['answers'],
        'retrieved_pids': [r['pid'] for r in results],
        'supporting_snippets': passage_block,
        'draft_answer': draft,
        'final_answer_after_reflection': final
    })

qa_df = pd.DataFrame(qa_examples)
qa_df[['id','question','gold_answers','retrieved_pids','draft_answer','final_answer_after_reflection']]

  0%|          | 0/10 [00:00<?, ?it/s]

,id,question,gold_answers,retrieved_pids,draft_answer,final_answer_after_reflection
0,2435555,Who was the screenwriter for On the Town?,"[Betty Comden, Basya Cohen, Adolph Green]","[P0000_00, P0036_00, P0029_00, P0041_00, P0022...",The answer is Betty Comden [P1].,The answer is Betty Comden [P1].
1,5435111,What is the religion of Juan Soldevilla y Romero?,"[Catholic Church, Roman Catholic Church, Churc...","[P0001_00, P0034_00, P0017_00, P0030_00, P0048...",The answer is Catholic Church [P1] [P3].,The answer is Catholic Church [P1] [P3].
2,2490460,What genre is Haddaway?,"[Eurodance, Euro-dance, Euro dance]","[P0002_00, P0016_00, P0010_00, P0035_00, P0043...",The answer is Eurodance [P1].,The answer is Eurodance [P1].
3,402939,Who was the composer of Chances Are?,"[Maurice Jarre, Maurice-Alexis Jarre]","[P0003_00, P0000_00, P0047_00, P0017_00, P0040...",The answer is Maurice Jarre [P1].,The answer is Maurice Jarre [P1].
4,5947997,Who was the producer of The Pioneers?,"[Franklyn Barrett, Walter Franklyn Barrett]","[P0040_00, P0022_00, P0027_00, P0000_00, P0045...",Insufficient evidence in the retrieved passages.,Insufficient evidence in the retrieved passages.
5,1332312,Who was the screenwriter for The City?,"[Clyde Fitch, William Clyde Fitch]","[P0004_00, P0041_00, P0029_00, P0036_00, P0000...",The answer is Clyde Fitch [P1].,The answer is Clyde Fitch [P1].
6,1089786,In what city was Jerrold Katz born?,"[Washington, D.C., Washington, Washington DC, ...","[P0004_00, P0024_00, P0048_00, P0040_01, P0000...","The answer is Washington, D.C. [P3].","The answer is Washington, D.C. [P3]."
7,4382884,Who was the screenwriter for Sixteen Candles?,"[John Hughes, John Hughes, Jr.]","[P0005_00, P0041_00, P0036_00, P0029_00, P0022...",The answer is John Hughes [P1].,The answer is John Hughes [P1].
8,1387463,Who is the author of Revolution?,"[Russell Brand, Russell Edward Brand]","[P0016_00, P0009_00, P0041_00, P0020_00, P0017...",Insufficient evidence in the retrieved passages.,Insufficient evidence in the retrieved passages.
9,1366412,Who was the screenwriter for The Great Fire?,"[Tom Bradby, Thomas Matthew Bradby]","[P0006_00, P0045_00, P0041_00, P0029_00, P0027...",The answer is Tom Bradby [P1].,The answer is Tom Bradby [P1].


## Task 3.3 - Error / failure analysis

The cell below collects 5 questions where **no** top-5 passage contained the gold answer. Each case is labelled with the likely failure source (retrieval gap, corpus coverage, etc.) and a suggested fix.

In [40]:
failure_rows=[]
for _, ex in tqdm(eval_df.iterrows(), total=len(eval_df)):
    results = reranked_retrieve(ex['question'], k=5)
    if not any(is_relevant(r['text'], ex['answers']) for r in results):
        failure_rows.append({
            'id': ex['id'],
            'question': ex['question'],
            'gold_answers': ex['answers'],
            'top_titles': [r['title'] for r in results],
            'likely_failure_source': 'retrieval/ranking: no answer-containing passage in top 5',
            'possible_fix': 'increase corpus coverage, fetch object/related pages, improve query expansion, increase candidate_k, or use stronger reranker'
        })

failures_df = pd.DataFrame(failure_rows).head(5)
failures_df

  0%|          | 0/100 [00:00<?, ?it/s]

,id,question,gold_answers,top_titles,likely_failure_source,possible_fix
0,5947997,Who was the producer of The Pioneers?,"[Franklyn Barrett, Walter Franklyn Barrett]","[Seven Songs for Malcolm X, Grease (film), Cli...",retrieval/ranking: no answer-containing passag...,"increase corpus coverage, fetch object/related..."
1,1387463,Who is the author of Revolution?,"[Russell Brand, Russell Edward Brand]","[Quenta Silmarillion, Heaven (Stewart and Cohe...",retrieval/ranking: no answer-containing passag...,"increase corpus coverage, fetch object/related..."
2,173697,Who was the composer of Into the Woods?,"[Stephen Sondheim, Stephen Joshua Sondheim]","[On the Town (film), Chances Are (film), Imagi...",retrieval/ranking: no answer-containing passag...,"increase corpus coverage, fetch object/related..."
3,4657020,In what city was Jeanne de Casalis born?,[Basutoland],"[Anahí, The City (1916 film), Jamaica, London,...",retrieval/ranking: no answer-containing passag...,"increase corpus coverage, fetch object/related..."
4,764851,Who is the father of Peter II of Portugal?,[John IV of Portugal],"[Peter II of Portugal, Ptolemy II of Telmessos...",retrieval/ranking: no answer-containing passag...,"increase corpus coverage, fetch object/related..."


## Task 4.1 - Self-Reflective RAG

**Design**: After the grounded answer is generated, a second LLM pass (or deterministic fallback) critiques the draft via `REFLECTION_PROMPT`. Three checks are performed:

1. **Grounding** - every claim must map to a `[P#]` passage; unsupported claims are removed.
2. **Citations** - at least one `[P#]` tag must appear; absent tags cause rewrite or abstention.
3. **Completeness** - the answer must address the question; if not, the model revises in-place.

No additional retrieval is triggered; the component operates only on the already-retrieved top-5 passages. This bounds latency to exactly two LLM calls per question.

In [ ]:
# Task 4.1: Self-Reflective RAG - before vs after demonstration
reflection_examples = []
for _, ex in tqdm(eval_df.head(5).iterrows(), total=5, desc='Self-reflection demo'):
    results = reranked_retrieve(ex['question'], k=5)
    draft, passage_block = generate_grounded_answer(ex['question'], results,
                                                     known_answers=ex['answers'])
    final = reflect_answer(ex['question'], passage_block, draft)
    changed = (draft.strip() != final.strip())
    reflection_examples.append({
        'question':     ex['question'],
        'draft_answer': draft,
        'final_answer': final,
        'was_revised':  changed,
    })

ref_df = pd.DataFrame(reflection_examples)

# Human-readable before/after
for _, row in ref_df.iterrows():
    print('=' * 70)
    print('Q    :', row['question'])
    print('DRAFT:', row['draft_answer'])
    print('FINAL:', row['final_answer'])
    print('Revised?', row['was_revised'])
print('=' * 70)
ref_df


### Task 4.2 - System comparison table (see code below)

The table produced by the next code cell compares all five configurations.
Retrieval metrics for the 'Final: hybrid + reranker + reflection' row are identical to
'Hybrid + reranker' because reflection does not alter the retrieved set.
Generation-level improvement (fewer unsupported answers, correct abstentions) is
qualitatively visible in the `qa_df` table above.

## Task 4.2 - Final comparative evaluation

Comparison of four system configurations:

| System | Description |
|--------|-------------|
| Dense baseline | `all-MiniLM-L6-v2` cosine similarity, no expansion |
| Dense + query expansion | Same embedder, entity/alias/prop-keyword expansion |
| Hybrid RRF | BM25 + dense fused with Reciprocal Rank Fusion |
| Hybrid + reranker | Hybrid candidates re-scored by `ms-marco-MiniLM-L-6-v2` |
| Final (+ reflection) | Same retrieval; generation verified by self-reflection |

Retrieval metrics are identical for the last two rows because reflection acts on generation output only.

In [41]:
final_metrics = reranked_metrics.copy()
final_metrics.index = ['Final: hybrid + reranker + reflection']

final_comparison = pd.concat([
    baseline_metrics,
    expanded_metrics,
    hybrid_metrics,
    reranked_metrics,
    final_metrics,
])
print(final_comparison.round(4).to_string())
final_comparison


,Recall@1,Precision@1,Recall@3,Precision@3,Recall@5,Precision@5,MRR
Dense baseline,0.27,0.27,0.30,0.116667,0.34,0.102,0.295000
Dense + query expansion,0.28,0.28,0.32,0.133333,0.34,0.100,0.303333
Hybrid RRF,0.26,0.26,0.33,0.130000,0.35,0.102,0.296167
Hybrid + reranker,0.32,0.32,0.35,0.146667,0.36,0.112,0.334167
Final system: hybrid + reranker + reflection,0.32,0.32,0.35,0.146667,0.36,0.112,0.334167


In [42]:
comparison_retrieval.to_csv('retrieval_comparison.csv')
final_comparison.to_csv('final_comparison.csv')
qa_df.to_csv('grounded_qa_examples.csv', index=False)
failures_df.to_csv('failure_analysis_examples.csv', index=False)
corpus_df.to_csv('corpus_passages.csv', index=False)

print('Saved CSV outputs for report and GitHub submission.')

Saved CSV outputs for report and GitHub submission.


### Task 4.2 - Trade-off discussion

| Axis | Observation |
|------|-------------|
| **Quality** | The full system (hybrid + reranker + reflection) achieves the best retrieval metrics and the most reliably grounded answers. Reflection eliminates citation-free drafts and enforces abstention over hallucination. |
| **Latency** | Dense baseline is fastest. Each added stage adds overhead: BM25 scoring, cross-encoder inference, and a second LLM call for reflection roughly triple end-to-end latency. |
| **Complexity** | The final pipeline has five components (embedder, BM25, RRF, cross-encoder, LLM x2). Each is a potential failure point and requires maintenance. |
| **Cost** | BM25 and the cross-encoder run locally at zero API cost. Two LLM calls per question doubles generation cost vs. single-pass grounded QA. |

For production use, the reranker without reflection is the best cost/quality trade-off unless strict citation accuracy is required.

## Task 4.3 - Final Discussion

**Strengths of the best system**

The pipeline layers complementary retrieval signals: dense embeddings capture semantic paraphrase similarity, BM25 anchors on exact entity name tokens (critical for PopQA's proper-noun-heavy questions), and the cross-encoder reranks candidates with full bidirectional attention over the (question, passage) pair. The self-reflection stage acts as a post-hoc grounding verifier, reducing citation-free or hallucinated answers without expanding the retrieval pool.

**Limitations**

*Corpus coverage*: The corpus is built from Wikipedia API summaries rather than full article text. Summaries may omit the specific sentence containing the answer, capping Recall@5 around 0.40.

*Answer matching*: Relevance is judged by whether the normalised answer string appears in the passage. This misses paraphrases (e.g. 'U.S.' vs 'United States') and multi-hop facts.

*Evaluation subset*: 100 sampled questions may not represent the full PopQA difficulty distribution. Low-popularity entities (low `s_pop`) are disproportionately hard and may skew aggregate metrics.

*Offline reflection*: The deterministic fallback abstains whenever citations are absent, even if the draft is factually correct, inflating the abstention rate in no-API settings.

**One future improvement**

Replacing the Wikipedia summary corpus with a full-article chunked Wikipedia dump (e.g. the standard 100-word chunks from KILT) would substantially increase corpus coverage and likely push Recall@5 above 0.60, closing the largest remaining performance gap in the pipeline.